# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower().str.replace(" ", "_")
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")
df["month"] = df["effective_to_date"].dt.month_name()
df.columns.tolist()   # check the names match what's used below

C:\Users\ovohj\AppData\Local\Temp\ipykernel_26652\3981030601.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")


['unnamed:_0',
 'customer',
 'state',
 'customer_lifetime_value',
 'response',
 'coverage',
 'education',
 'effective_to_date',
 'employmentstatus',
 'gender',
 'income',
 'location_code',
 'marital_status',
 'monthly_premium_auto',
 'months_since_last_claim',
 'months_since_policy_inception',
 'number_of_open_complaints',
 'number_of_policies',
 'policy_type',
 'policy',
 'renew_offer_type',
 'sales_channel',
 'total_claim_amount',
 'vehicle_class',
 'vehicle_size',
 'vehicle_type',
 'month']

In [2]:
low_claim_yes = df[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")]
print(low_claim_yes.shape)
low_claim_yes.head()

(1399, 27)


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,January
8,8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,...,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN,January
15,15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,...,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN,January
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,...,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A,February
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,...,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN,February


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [3]:
yes = df[df["response"] == "Yes"]

segments = (
    yes.groupby(["policy_type", "gender"])
    .agg(
        avg_monthly_premium=("monthly_premium_auto", "mean"),
        avg_clv=("customer_lifetime_value", "mean"),
        avg_claim=("total_claim_amount", "mean"),
        customers=("customer", "count"),
    )
    .round(2)
)
segments["premium_minus_claim_ratio"] = (
    segments["avg_monthly_premium"] / segments["avg_claim"]
).round(3)
segments.sort_values("avg_clv", ascending=False)

avg_monthly_premium  avg_clv  avg_claim  customers  \
policy_type    gender                                                       
Personal Auto  F                     99.00  8339.79     452.97        540   
Special Auto   M                     86.34  8247.09     429.53         32   
Corporate Auto M                     92.19  7944.47     408.58        154   
               F                     94.30  7712.63     433.74        169   
Special Auto   F                     92.31  7691.58     453.28         35   
Personal Auto  M                     91.09  7448.38     457.01        536   

                       premium_minus_claim_ratio  
policy_type    gender                             
Personal Auto  F                           0.219  
Special Auto   M                           0.201  
Corporate Auto M                           0.226  
               F                           0.217  
Special Auto   F                           0.204  
Personal Auto  M                           0.199

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [4]:
state_counts = df.groupby("state")["customer"].count()
state_counts[state_counts > 500].sort_values(ascending=False)

state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: customer, dtype: int64

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [5]:
(
    df.groupby(["education", "gender"])["customer_lifetime_value"]
    .agg(["max", "min", "median"])
    .round(2)
)

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [6]:
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

state_month = pd.pivot_table(
    df, index="state", columns="month",
    values="customer", aggfunc="count", fill_value=0,
)
state_month = state_month.reindex(
    columns=[m for m in month_order if m in state_month.columns]
)
state_month

month,January,February
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [7]:
top3 = df["state"].value_counts().head(3).index

top3_by_month = (
    df[df["state"].isin(top3)]
    .groupby(["state", "month"])["customer"].count()
    .reset_index(name="policies_sold")
    .sort_values(["state", "policies_sold"], ascending=[True, False])
)
top3_by_month

,state,month,policies_sold
1,Arizona,January,1008
0,Arizona,February,929
3,California,January,1918
2,California,February,1634
5,Oregon,January,1565
4,Oregon,February,1344


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [8]:
# 1. Response flag: 1 if "Yes", 0 otherwise
df["responded"] = (df["response"] == "Yes").astype(int)

# 2. Wide table: one column per sales channel, with its response rate (%)
wide = (
    df.pivot_table(columns="sales_channel", values="responded", aggfunc="mean")
    .mul(100)
    .round(2)
)
print(wide)

# 3. Unpivot to long format: one row per channel
response_rate = (
    wide.reset_index(drop=True)
    .melt(var_name="sales_channel", value_name="response_rate_pct")
    .sort_values("response_rate_pct", ascending=False)
    .reset_index(drop=True)
)

# 4. Add the number of customers and "Yes" responses for context
extra = (
    df.groupby("sales_channel")
    .agg(customers=("customer", "count"), yes_responses=("responded", "sum"))
    .reset_index()
)
response_rate = response_rate.merge(extra, on="sales_channel")
response_rate

sales_channel  Agent  Branch  Call Center    Web
responded      18.01   10.79        10.32  10.89


,sales_channel,response_rate_pct,customers,yes_responses
0,Agent,18.01,4121,742
1,Web,10.89,1626,177
2,Branch,10.79,3022,326
3,Call Center,10.32,2141,221
